<style>
.jp-Notebook, .notebook-container, .markdown-body {font-family: Arial, Helvetica, sans-serif;}
.jp-MarkdownOutput, .text_cell_render {font-size: 18px; line-height: 1.65;}
h1 {font-size: 2.25rem !important; margin-top: 0.35em !important;}
h2 {font-size: 1.65rem !important; margin-top: 1.35em !important;}
h3 {font-size: 1.25rem !important; margin-top: 1.1em !important;}
table {font-size: 0.95em;}
blockquote {border-left: 4px solid #aaa; padding-left: 1rem;}
</style>


# 02 · Centro, dispersión y relaciones

<p><a href="https://colab.research.google.com/github/mauriciorslrv/DS_basics/blob/content/statistics-module-rework/02%20An%C3%A1lisis%20Estad%C3%ADstico%20de%20Datos/notebooks/02_Medidas_Descriptivas_y_Relaciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a></p>

> **Objetivo:** describir un conjunto con centro, dispersión, forma y relaciones sin reducirlo a un solo promedio.


## 1. Un promedio puede engañar

Una descripción útil suele responder:

**centro → dispersión → forma → valores extremos → relación con otras variables**


<img src="https://raw.githubusercontent.com/mauriciorslrv/DS_basics/content/statistics-module-rework/02%20An%C3%A1lisis%20Estad%C3%ADstico%20de%20Datos/imgs/Outliers.jpeg" alt="Valores atípicos" width="620">


### ¿Por qué este ejercicio?
Usaremos tiempos de entrega con un valor extremo. El objetivo es ver cómo **el mismo dato puede alterar unas métricas mucho más que otras** y aprender a no eliminar un outlier automáticamente.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

serie = pd.Series([28,31,32,33,34,35,36,37,39,40,42,78], name="tiempo_min")
serie.describe()


## 2. Media vs. mediana

La **media** usa todos los valores y es sensible a extremos. La **mediana** depende del orden y suele ser más robusta.

La pregunta no es cuál es "mejor" en abstracto, sino **cuál representa mejor el fenómeno que quieres comunicar**.


In [ ]:
sin_extremo = serie[serie < 60]
print("Media con/sin extremo:", serie.mean(), sin_extremo.mean())
print("Mediana con/sin extremo:", serie.median(), sin_extremo.median())


### 💡 IDEA
No elimines un outlier sólo porque se ve raro. Primero pregunta si es:

- un error de captura;
- un evento real pero raro;
- o justamente el riesgo que necesitas estudiar.


## 3. Mismo promedio, distinta historia

Este ejemplo es pequeño pero poderoso: dos conjuntos pueden compartir la misma media y tener comportamientos muy diferentes.


In [ ]:
a = np.array([8, 9, 10, 11, 12])
b = np.array([0, 5, 10, 15, 20])

comparacion = pd.DataFrame({
    "serie": ["A", "B"],
    "media": [a.mean(), b.mean()],
    "desviacion": [a.std(ddof=1), b.std(ddof=1)],
    "rango": [a.max()-a.min(), b.max()-b.min()]
})
comparacion


**Interpretación:** ambas tienen media 10, pero B es mucho más dispersa. Si sólo reportáramos la media, perderíamos esa diferencia.


## 4. Dispersión

- **Rango:** distancia entre mínimo y máximo.
- **IQR:** amplitud del 50% central de los datos.
- **Varianza / desviación estándar:** dispersión alrededor de la media.
- **Coeficiente de variación (CV):** dispersión relativa respecto a la media.


In [ ]:
q1, q3 = serie.quantile([0.25,0.75])
iqr = q3-q1

resumen = {
    "rango": serie.max()-serie.min(),
    "IQR": iqr,
    "varianza_muestral": serie.var(ddof=1),
    "desviacion_estandar": serie.std(ddof=1),
    "CV": serie.std(ddof=1)/serie.mean()
}
pd.Series(resumen)


In [ ]:
# Regla exploratoria de 1.5 × IQR: marca candidatos a outlier, no "errores".
lim_inf, lim_sup = q1-1.5*iqr, q3+1.5*iqr
serie[(serie<lim_inf)|(serie>lim_sup)]


In [ ]:
plt.boxplot(serie, vert=False)
plt.xlabel("Minutos")
plt.title("Centro, dispersión y posibles outliers")
plt.show()


## 5. Forma y relación

La **asimetría** describe si una distribución tiene una cola más pronunciada hacia un lado.

La **correlación** resume asociación lineal entre dos variables numéricas. No explica por sí sola por qué ocurre esa asociación.


In [ ]:
print(f"Asimetría: {serie.skew():.2f}")


### ¿Por qué este segundo ejercicio?
Creamos pedidos con más productos y tiempos algo mayores. Sabemos cómo fueron generados, así que podemos usarlo para leer una correlación sin confundirla con causalidad.


In [ ]:
rng=np.random.default_rng(7)
pedidos=pd.DataFrame({"productos":rng.integers(1,8,80)})
pedidos["tiempo_min"]=24+3.2*pedidos["productos"]+rng.normal(0,5,80)

print(f"Correlación: {pedidos['productos'].corr(pedidos['tiempo_min']):.2f}")

plt.scatter(pedidos["productos"],pedidos["tiempo_min"],alpha=.7)
plt.xlabel("Productos")
plt.ylabel("Minutos")
plt.show()


### ⚠️ ERROR ÚTIL
Correlación describe asociación lineal; por sí sola **no demuestra causalidad**.


## 6. Medias especiales: sólo cuando el problema lo pide

- **Geométrica:** útil para crecimiento compuesto.
- **Armónica:** útil para promediar ciertas tasas, como velocidades sobre distancias iguales.

No hace falta memorizar una colección de medias: conviene entender **qué estructura matemática tiene el problema**.


In [ ]:
from scipy.stats import gmean, hmean

print("Factor geométrico:", gmean([1.05,1.10,0.97]))
print("Velocidad armónica:", hmean([60,40]))


## 7. Mini reto

Añade otro valor extremo a la serie y compara media, mediana, desviación estándar e IQR.

Después responde: **si sólo pudieras comunicar dos métricas, cuáles elegirías y por qué?**


## 📚 Material adicional
- [NIST · Exploratory Data Analysis](https://www.itl.nist.gov/div898/handbook/eda/eda.htm) — centro, dispersión, outliers y forma desde una perspectiva práctica.
- [SciPy · Statistics](https://docs.scipy.org/doc/scipy/tutorial/stats.html) — funciones estadísticas y ejemplos.
- [OpenIntro Statistics](https://www.openintro.org/book/os/) — capítulos de exploración de datos y variabilidad.

### 🧾 Términos clave
**media · mediana · IQR · desviación estándar · coeficiente de variación · outlier · asimetría · correlación**


## Qué sigue
Ahora pasamos de **lo observado** a **lo que podría ocurrir** mediante modelos de probabilidad.
